# Objective

This notebook defines the core profiling data model and the foundational APIs used by the distributed profiling pipeline.

The goal is to establish a clean, extensible representation for profiling events before introducing multithreading, batching, networking, storage, and performance analysis.

The notebook focuses on four main objectives:

- Define the initial profiling event types, including kernel execution events and memory-transfer events.
- Represent heterogeneous profiling events using a unified `ProfileEvent` type based on modern C++20 features.
- Design lightweight interfaces for event sources, serializers, and data sinks so that major system components remain loosely coupled.
- Validate the design with a small standalone C++20 program and CMake-based build.

The initial architecture developed in this notebook is:

```text
Event Source
     ↓
ProfileEvent
     ↓
Future Processing Pipeline
     ↓
Serializer
     ↓
Transport / Storage
```

At this stage, the project does not yet implement multithreading or networking. The focus is on defining stable data structures and interfaces that later components can reuse without depending on specific profiling tools or storage formats.

The design should make it possible to replace or extend individual components independently. For example:

```text
SyntheticEventSource
        ↓
     ProfileEvent
        ↓
      Pipeline
```

can later become:

```text
NsightEventSource
        ↓
     ProfileEvent
        ↓
      Pipeline
```

without requiring changes to the queue, networking, or storage layers.

Similarly, the same profiling events should eventually support different serialization and storage implementations:

```text
                ┌── JSON Serializer
ProfileEvent ───┼── Binary Serializer
                └── Future Serializer
```

and:

```text
              ┌── Console Sink
ProfileEvent ─┼── File Sink
              └── SQLite Sink
```

By the end of this notebook, the project should have a small but production-oriented C++20 foundation that can serve as the common data layer for the rest of the distributed profiling infrastructure.

## Standalone Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The setup section installs and verifies the minimal toolchain required for the C++20 implementation in this notebook.

The required environment includes:

- Linux runtime
- GCC / G++ with C++20 support
- CMake
- Ninja
- Git

This notebook does not require CUDA, networking libraries, databases, or external serialization libraries yet.

The goal of the setup stage is to ensure that the core C++20 data model and API design can be compiled and tested in isolation.

### Environment Preparation

The setup performs the following steps:

1. Update the package index.
2. Install the required C++ build tools.
3. Verify compiler and build-system versions.
4. Create the project directory structure used by this notebook.
5. Confirm that a basic C++20 program can be compiled successfully.

The project structure created in this notebook is:

```text
distributed-profiler/
├── include/
├── src/
├── tests/
└── CMakeLists.txt
```

Later notebooks will reuse and extend this structure with additional modules such as networking, storage, benchmarks, and profiling adapters.

In [2]:
!sudo apt-get update -qq

!sudo apt-get install -y \
    build-essential \
    g++ \
    cmake \
    ninja-build \
    git

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.10ubuntu1).
g++ is already the newest version (4:13.2.0-7ubuntu1).
g++ set to manually installed.
cmake is already the newest version (3.28.3-1build7).
git is already the newest version (1:2.43.0-1ubuntu7.3).
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 129 kB of archives.
After this operation, 364 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 ninja-build amd64 1.11.1-2 [129 kB]
Fetched 129 kB in 0s (283 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so

In [3]:
!echo "=== Compiler ==="
!g++ --version | head -n 1

!echo ""
!echo "=== CMake ==="
!cmake --version | head -n 1

!echo ""
!echo "=== Ninja ==="
!ninja --version

!echo ""
!echo "=== Git ==="
!git --version

=== Compiler ===
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0

=== CMake ===
cmake version 3.31.10

=== Ninja ===
1.11.1

=== Git ===
git version 2.43.0


In [4]:
!rm -rf distributed-profiler

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests

!find distributed-profiler -maxdepth 2 -type d

distributed-profiler
distributed-profiler/include
distributed-profiler/tests
distributed-profiler/src


In [5]:
%%writefile cpp20_sanity_check.cpp

#include <iostream>
#include <optional>
#include <span>
#include <variant>
#include <vector>

int main() {
    std::vector<int> values{1, 2, 3};

    std::span<int> view(values);

    std::variant<int, std::string> value =
        std::string("C++20 ready");

    std::optional<int> result = 42;

    std::cout << std::get<std::string>(value) << '\n';
    std::cout << "span size: " << view.size() << '\n';
    std::cout << "optional value: " << *result << '\n';

    return 0;
}

Writing cpp20_sanity_check.cpp


In [6]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    cpp20_sanity_check.cpp \
    -o cpp20_sanity_check

!./cpp20_sanity_check

C++20 ready
span size: 3
optional value: 42


## Core Data Model and API Design

This section implements the foundational data structures and interfaces used by the profiling pipeline.

Before introducing multithreading, networking, batching, or storage, the system needs a stable representation for profiling data and a clear contract between major components.

The implementation in this section focuses on three layers:

```text
Profiling Event Types
        ↓
Unified ProfileEvent
        ↓
Core Interfaces
```

### 1. Profiling Event Types

The first step is to define the profiling information that flows through the system.

The initial implementation supports two event types:

```text
KernelEvent
MemoryEvent
```

`KernelEvent` represents a GPU kernel execution and contains information such as:

```text
timestamp
duration
device_id
kernel_name
```

`MemoryEvent` represents a memory-transfer operation and contains information such as:

```text
timestamp
device_id
bytes
operation
```

These two event types are intentionally simple.

The goal is not to model every possible profiler field at the beginning of the project. Instead, the initial data model should contain enough information to exercise the entire pipeline while remaining easy to understand and extend.

---

### 2. Unified `ProfileEvent`

The rest of the system should not require separate processing pipelines for every event type.

For example, we do not want:

```text
KernelEvent → KernelQueue → KernelSerializer

MemoryEvent → MemoryQueue → MemorySerializer
```

Instead, both event types should enter the same pipeline:

```text
KernelEvent ─┐
             ├── ProfileEvent
MemoryEvent ─┘
                    ↓
                Pipeline
```

The project therefore uses:

```cpp
std::variant<KernelEvent, MemoryEvent>
```

to define a common `ProfileEvent` type.

This gives the system a type-safe way to store heterogeneous profiling events without using raw pointers or a large inheritance hierarchy.

Later components such as the queue, batcher, serializer, transport, and storage layer will all operate on `ProfileEvent`.

---

### 3. Event Source Interface

The profiling pipeline should not depend on where the events originate.

During early development, events may be generated synthetically:

```text
SyntheticEventSource
```

Later, they may come from real profiling tools:

```text
NsightEventSource
ROCmEventSource
```

All of these sources should expose a common interface.

Conceptually:

```text
SyntheticEventSource ─┐
NsightEventSource ─────┼── IEventSource → ProfileEvent
ROCmEventSource ───────┘
```

This allows the event-generation mechanism to change without requiring modifications to the rest of the pipeline.

---

### 4. Serializer Interface

Before profiling events can be transferred across a network, they must be converted into a byte representation.

Different serialization strategies may be used later:

```text
JSON
Binary
Future compact format
```

The pipeline therefore introduces an `ISerializer` interface.

Conceptually:

```text
ProfileEvent
     ↓
ISerializer
     ↓
byte buffer
```

This separates the event representation from the encoding format.

Later experiments will compare different serialization approaches in terms of:

```text
encoded size
CPU overhead
throughput
latency
```

---

### 5. Data Sink Interface

The collector eventually needs somewhere to send processed profiling events.

During development, the destination may simply be:

```text
Console
```

Later it may become:

```text
File
SQLite
```

These implementations should share a common `IDataSink` interface.

Conceptually:

```text
ProfileEvent
     ↓
IDataSink
     ├── ConsoleSink
     ├── FileSink
     └── SQLiteSink
```

This makes it possible to change the storage backend independently of the networking and event-processing layers.

---

## Design Goal

The main design goal of this section is loose coupling.

Each component should depend primarily on a small interface rather than on a specific implementation.

For example:

```text
Event Source
    ↓
ProfileEvent
    ↓
Queue
    ↓
Serializer
    ↓
Transport
```

The queue should not need to know whether the event came from Nsight or a synthetic generator.

The transport should not need to know how the event was generated.

The storage layer should not need to know which network protocol delivered the event.

This separation will make later stages of the project easier to extend, test, benchmark, and refactor.

---

## Files Implemented in This Section

This section will create the following files:

```text
distributed-profiler/
├── include/
│   ├── profile_event.hpp
│   ├── event_source.hpp
│   ├── serializer.hpp
│   └── data_sink.hpp
│
├── src/
│
├── tests/
│   └── test_profile_event.cpp
│
└── CMakeLists.txt
```

The implementation order will be:

```text
1. KernelEvent
2. MemoryEvent
3. ProfileEvent
4. std::visit test
5. IEventSource
6. ISerializer
7. IDataSink
8. CMake build
9. Basic validation
```

By the end of this section, the project should have a small but reusable C++20 core that later notebooks can build upon.

In [7]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

// Define a struct named KernelEvent.

// fields:
// - timestamp
// - duration
// - device_id
// - kernel_name
struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};


// Define a struct named MemoryEvent.
//
// fields:
// - timestamp
// - device_id
// - bytes
// - operation
struct MemoryEvent {
  std::uint64_t timestamp_ns;
  std::uint32_t device_id;
  std::uint64_t bytes;
  std::string operation;
};


// Define a unified ProfileEvent type using std::variant.
//
// It should be able to hold either:
// - KernelEvent
// - MemoryEvent
using ProfileEvent = std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [8]:
%%writefile distributed-profiler/include/event_source.hpp

#pragma once

#include "profile_event.hpp"

#include <optional>

class IEventSource {
public:
    virtual ~IEventSource() = default;

    virtual std::optional<ProfileEvent> next_event() = 0;
};

Writing distributed-profiler/include/event_source.hpp


In [9]:
%%writefile distributed-profiler/include/serializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <vector>

class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};

Writing distributed-profiler/include/serializer.hpp


In [10]:
%%writefile distributed-profiler/include/data_sink.hpp

#pragma once

#include "profile_event.hpp"

class IDataSink {
public:
    virtual ~IDataSink() = default;

    virtual void write(const ProfileEvent& event) = 0;
};

Writing distributed-profiler/include/data_sink.hpp


In [11]:
%%writefile distributed-profiler/tests/test_profile_event.cpp

#include "profile_event.hpp"

#include <iostream>
#include <type_traits>

int main() {
    KernelEvent kernel{
        1000,
        200,
        0,
        "gemm_kernel"
    };

    MemoryEvent memory{
        1500,
        0,
        4096,
        "H2D"
    };

    ProfileEvent event1 = kernel;
    ProfileEvent event2 = memory;

    auto print_event = [](const ProfileEvent& event) {
        std::visit(
            [](const auto& e) {
                using T = std::decay_t<decltype(e)>;

                if constexpr (std::is_same_v<T, KernelEvent>) {
                    std::cout
                        << "[KernelEvent] "
                        << "device=" << e.device_id
                        << ", kernel=" << e.kernel_name
                        << ", timestamp=" << e.timestamp_ns
                        << ", duration=" << e.duration_ns
                        << " ns\n";
                }
                else if constexpr (std::is_same_v<T, MemoryEvent>) {
                    std::cout
                        << "[MemoryEvent] "
                        << "device=" << e.device_id
                        << ", operation=" << e.operation
                        << ", bytes=" << e.bytes
                        << ", timestamp=" << e.timestamp_ns
                        << " ns\n";
                }
            },
            event
        );
    };

    print_event(event1);
    print_event(event2);

    return 0;
}

Writing distributed-profiler/tests/test_profile_event.cpp


In [12]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/tests/test_profile_event.cpp \
    -o test_profile_event

In [13]:
!./test_profile_event

[KernelEvent] device=0, kernel=gemm_kernel, timestamp=1000, duration=200 ns
[MemoryEvent] device=0, operation=H2D, bytes=4096, timestamp=1500 ns


## Synthetic Event Source

Before integrating real profiling data from tools such as Nsight or ROCm, the project first introduces a synthetic event source.

The purpose of the synthetic source is to generate controlled profiling events without depending on a GPU profiler, trace file, or external runtime.

This allows the rest of the profiling pipeline to be developed and tested independently.

The synthetic source implements the common `IEventSource` interface:

```text
             IEventSource
                  ↑
                  │
      SyntheticEventSource
```

The interface exposes a single operation:

```cpp
next_event()
```

Each call returns the next available profiling event.

Conceptually:

```text
SyntheticEventSource
        ↓
    next_event()
        ↓
   ProfileEvent
```

The source may generate events such as:

```text
KernelEvent("gemm_kernel")
MemoryEvent("H2D")
KernelEvent("softmax_kernel")
MemoryEvent("D2H")
```

These events use the same `ProfileEvent` representation that will later be used by real profiling backends.

---

### Why Use a Synthetic Source?

Without a synthetic source, development of the rest of the system would depend on the availability and correctness of a real profiling backend.

For example:

```text
GPU Workload
    ↓
Nsight
    ↓
Trace Parsing
    ↓
ProfileEvent
    ↓
Pipeline
```

If trace collection or parsing fails, development of the queue, serialization, networking, and storage layers would also be blocked.

The synthetic source removes this dependency:

```text
SyntheticEventSource
        ↓
   ProfileEvent
        ↓
Queue / Serializer / TCP / Storage
```

This makes it possible to isolate and validate each infrastructure component before introducing real GPU profiling data.

---

### Expected Behavior

The initial implementation stores a small sequence of predefined profiling events internally.

Each call to:

```cpp
next_event()
```

returns one event from that sequence.

For example:

```text
call 1 → KernelEvent
call 2 → MemoryEvent
call 3 → KernelEvent
call 4 → MemoryEvent
```

Once all events have been consumed:

```cpp
next_event()
```

returns:

```cpp
std::nullopt
```

This indicates that the event source has no more data available.

---

### Design Goal

The main design goal is interchangeability.

The rest of the profiling system should depend only on `IEventSource`, not on a specific implementation.

This means the pipeline can initially use:

```text
SyntheticEventSource
```

and later replace it with:

```text
NsightEventSource
```

or:

```text
ROCmEventSource
```

without changing the downstream queue, serializer, transport, or storage components.

Conceptually:

```text
SyntheticEventSource ─┐
NsightEventSource ─────┼── IEventSource → ProfileEvent → Pipeline
ROCmEventSource ───────┘
```

---

### Concepts Practiced

This implementation introduces several important modern C++ concepts:

```text
inheritance
virtual functions
override
std::optional
std::vector
std::variant
interface-based design
```

It also provides the first concrete implementation of the abstraction layer defined earlier in the notebook.

---

### Completion Criteria

This section is complete when:

```text
[ ] SyntheticEventSource inherits from IEventSource
[ ] It stores multiple predefined ProfileEvent objects
[ ] next_event() returns one event at a time
[ ] next_event() returns std::nullopt after all events are consumed
[ ] KernelEvent and MemoryEvent can both be produced
[ ] The implementation compiles and runs successfully
```

Once this is working, the next step will be to connect the generated events to a concrete serializer and data sink.

In [14]:
%%writefile distributed-profiler/include/synthetic_event_source.hpp

#pragma once

#include "event_source.hpp"

#include <cstddef>
#include <optional>
#include <vector>

class SyntheticEventSource : public IEventSource {
public:
    SyntheticEventSource();

    std::optional<ProfileEvent> next_event() override;

private:
    std::vector<ProfileEvent> events_;

    std::size_t current_index_;
};

Writing distributed-profiler/include/synthetic_event_source.hpp


In [15]:
%%writefile distributed-profiler/src/synthetic_event_source.cpp

#include "synthetic_event_source.hpp"

SyntheticEventSource::SyntheticEventSource()
    : current_index_(0) {

    // Add several predefined profiling events into events_.
    events_.push_back(
        KernelEvent{
            1000,
            200,
            0,
            "gemm_kernel"
        }
    );
    events_.push_back(
        MemoryEvent{
            1500,
            0,
            4096,
            "H2D"
        }
    );
    events_.push_back(
        KernelEvent{
            2000,
            150,
            0,
            "softmax_kernel"
        }
    );

    events_.push_back(
        MemoryEvent{
            2300,
            0,
            8192,
            "D2H"
        }
    );
}



// Return one ProfileEvent at a time.
//
// if there are still events:
//     return the current event
//     increment current_index_
//
// otherwise:
//     return std::nullopt
//
std::optional<ProfileEvent>
SyntheticEventSource::next_event() {

    if (current_index_ < events_.size()) {
        return events_[current_index_++];
    }

    return std::nullopt;

}

Writing distributed-profiler/src/synthetic_event_source.cpp


In [16]:
%%writefile distributed-profiler/tests/test_synthetic_event_source.cpp

#include "synthetic_event_source.hpp"

#include <iostream>
#include <type_traits>

int main() {
    SyntheticEventSource source;

    int count = 0;

    while (auto event = source.next_event()) {
        ++count;

        std::visit(
            [](const auto& e) {
                using T = std::decay_t<decltype(e)>;

                if constexpr (std::is_same_v<T, KernelEvent>) {
                    std::cout
                        << "[KernelEvent] "
                        << "kernel=" << e.kernel_name
                        << ", device=" << e.device_id
                        << ", timestamp=" << e.timestamp_ns
                        << ", duration=" << e.duration_ns
                        << " ns\n";
                }
                else if constexpr (std::is_same_v<T, MemoryEvent>) {
                    std::cout
                        << "[MemoryEvent] "
                        << "operation=" << e.operation
                        << ", device=" << e.device_id
                        << ", bytes=" << e.bytes
                        << ", timestamp=" << e.timestamp_ns
                        << " ns\n";
                }
            },
            *event
        );
    }

    std::cout << "Total events: " << count << '\n';

    auto extra_event = source.next_event();

    if (!extra_event.has_value()) {
        std::cout << "No more events: PASS\n";
    }
    else {
        std::cout << "Unexpected extra event: FAIL\n";
        return 1;
    }

    return 0;
}

Writing distributed-profiler/tests/test_synthetic_event_source.cpp


In [17]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/synthetic_event_source.cpp \
    distributed-profiler/tests/test_synthetic_event_source.cpp \
    -o test_synthetic_event_source

In [18]:
!./test_synthetic_event_source

[KernelEvent] kernel=gemm_kernel, device=0, timestamp=1000, duration=200 ns
[MemoryEvent] operation=H2D, device=0, bytes=4096, timestamp=1500 ns
[KernelEvent] kernel=softmax_kernel, device=0, timestamp=2000, duration=150 ns
[MemoryEvent] operation=D2H, device=0, bytes=8192, timestamp=2300 ns
Total events: 4
No more events: PASS


## Simple Text Serializer

The next step is to convert `ProfileEvent` objects into a transportable representation.

At this stage, the project uses a simple text-based serialization format rather than introducing an external JSON or binary serialization library.

The goal is to validate the `ISerializer` abstraction and create a representation that can later be sent over TCP.

The serialization flow is:

```text
ProfileEvent
    ↓
SimpleTextSerializer
    ↓
serialized byte buffer
```

For a `KernelEvent`, the output format will be:

```text
KERNEL|timestamp|duration|device_id|kernel_name
```

Example:

```text
KERNEL|1000|200|0|gemm_kernel
```

For a `MemoryEvent`, the output format will be:

```text
MEMORY|timestamp|device_id|bytes|operation
```

Example:

```text
MEMORY|1500|0|4096|H2D
```

The serializer will use `std::visit` to determine which concrete event type is currently stored inside `ProfileEvent`.

Conceptually:

```text
                 ┌── KernelEvent ──→ KERNEL|...
ProfileEvent ────┤
                 └── MemoryEvent ──→ MEMORY|...
```

The resulting text will then be converted into:

```cpp
std::vector<std::byte>
```

so that the serializer interface remains compatible with future networking and binary serialization implementations.

### Why Start with a Simple Text Format?

A text-based format is useful during the early development stage because it is:

- easy to inspect
- easy to debug
- easy to validate
- independent of third-party libraries
- sufficient for testing the serializer abstraction
- suitable for the first TCP transport implementation

The goal is not to design the final production wire format yet.

Later stages can introduce more efficient formats such as compact binary serialization without changing the rest of the pipeline.

### Design Goal

The rest of the profiling pipeline should depend only on:

```cpp
ISerializer
```

rather than on a specific serialization format.

This allows:

```text
SimpleTextSerializer
JSONSerializer
BinarySerializer
```

to be interchangeable implementations of the same interface.

### Completion Criteria

This section is complete when:

```text
[ ] SimpleTextSerializer inherits from ISerializer
[ ] KernelEvent can be serialized
[ ] MemoryEvent can be serialized
[ ] std::visit selects the correct serialization logic
[ ] serialized output is returned as std::vector<std::byte>
[ ] output format is validated with a small test
```

In [19]:
%%writefile distributed-profiler/include/simple_text_serializer.hpp

#pragma once

#include "serializer.hpp"

class SimpleTextSerializer : public ISerializer {
public:
    std::vector<std::byte>
    serialize(const ProfileEvent& event) const override;
};

Writing distributed-profiler/include/simple_text_serializer.hpp


In [20]:
%%writefile distributed-profiler/src/simple_text_serializer.cpp

#include "simple_text_serializer.hpp"

#include <string>
#include <type_traits>
#include <variant>

std::vector<std::byte>
SimpleTextSerializer::serialize(const ProfileEvent& event) const {

    std::string text;

    std::visit(
        [&](const auto& e) {
            using T = std::decay_t<decltype(e)>;

            if constexpr (std::is_same_v<T, KernelEvent>) {
                // Build:
                // KERNEL|timestamp|duration|device_id|kernel_name
                text =
                    "KERNEL|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.duration_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    e.kernel_name;
            }
            else if constexpr (std::is_same_v<T, MemoryEvent>) {
                // Build:
                // MEMORY|timestamp|device_id|bytes|operation
                text =
                    "MEMORY|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    std::to_string(e.bytes) + "|" +
                    e.operation;
            }
        },
        event
    );

    // Convert std::string into std::vector<std::byte>
    // and return it.
    std::vector<std::byte> bytes;

    bytes.reserve(text.size());// Here, `reserve()` is not required, but it can preallocate memory to avoid repeated `vector` reallocations during `push_back()`.


    for (char c : text) {
        bytes.push_back(
            static_cast<std::byte>(c)
        );
    }

    return bytes;
}

Writing distributed-profiler/src/simple_text_serializer.cpp


In [21]:
%%writefile distributed-profiler/tests/test_simple_text_serializer.cpp

#include "simple_text_serializer.hpp"

#include <cstddef>
#include <iostream>
#include <string>
#include <vector>

std::string bytes_to_string(const std::vector<std::byte>& bytes) {
    std::string result;
    result.reserve(bytes.size());

    for (std::byte b : bytes) {
        result.push_back(
            static_cast<char>(b)
        );
    }

    return result;
}

int main() {
    SimpleTextSerializer serializer;

    ProfileEvent kernel_event = KernelEvent{
        1000,
        200,
        0,
        "gemm_kernel"
    };

    ProfileEvent memory_event = MemoryEvent{
        1500,
        0,
        4096,
        "H2D"
    };

    auto kernel_bytes =
        serializer.serialize(kernel_event);

    auto memory_bytes =
        serializer.serialize(memory_event);

    std::string kernel_text =
        bytes_to_string(kernel_bytes);

    std::string memory_text =
        bytes_to_string(memory_bytes);

    std::cout
        << "Kernel serialized: "
        << kernel_text
        << '\n';

    std::cout
        << "Memory serialized: "
        << memory_text
        << '\n';

    if (kernel_text !=
        "KERNEL|1000|200|0|gemm_kernel") {

        std::cerr
            << "Kernel serialization: FAIL\n";

        return 1;
    }

    if (memory_text !=
        "MEMORY|1500|0|4096|H2D") {

        std::cerr
            << "Memory serialization: FAIL\n";

        return 1;
    }

    std::cout
        << "SimpleTextSerializer: PASS\n";

    return 0;
}

Writing distributed-profiler/tests/test_simple_text_serializer.cpp


In [22]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/tests/test_simple_text_serializer.cpp \
    -o test_simple_text_serializer

In [23]:
!./test_simple_text_serializer

Kernel serialized: KERNEL|1000|200|0|gemm_kernel
Memory serialized: MEMORY|1500|0|4096|H2D
SimpleTextSerializer: PASS


## Console Sink

The next step is to implement a simple concrete data sink.

A data sink represents the destination of processed profiling events.

The project defines a generic `IDataSink` interface so that different storage or output backends can be used without changing the rest of the pipeline.

The first implementation is:

```text
ConsoleSink
```

Its responsibility is simple:

```text
ProfileEvent
    ↓
ConsoleSink
    ↓
terminal output
```

The `ConsoleSink` is useful during early development because it provides an immediate way to verify that profiling events are flowing correctly through the system.

Later implementations may include:

```text
FileSink
SQLiteSink
```

while preserving the same `IDataSink` interface.

### Design Goal

The rest of the system should interact with:

```cpp
IDataSink
```

rather than depending directly on a specific storage backend.

This makes the sink implementation interchangeable:

```text
                ┌── ConsoleSink
ProfileEvent ───┼── FileSink
                └── SQLiteSink
```

### Completion Criteria

This section is complete when:

```text
[ ] ConsoleSink inherits from IDataSink
[ ] write() accepts a ProfileEvent
[ ] KernelEvent is printed correctly
[ ] MemoryEvent is printed correctly
[ ] std::visit is used to select the correct output logic
[ ] the implementation compiles and runs successfully
```

In [24]:
%%writefile distributed-profiler/include/console_sink.hpp

#pragma once

#include "data_sink.hpp"

class ConsoleSink : public IDataSink {
public:
    void write(const ProfileEvent& event) override;
};

Writing distributed-profiler/include/console_sink.hpp


In [25]:
%%writefile distributed-profiler/src/console_sink.cpp

#include "console_sink.hpp"

#include <iostream>
#include <type_traits>
#include <variant>

void ConsoleSink::write(const ProfileEvent& event) {

    std::visit(
        [](const auto& e) {
            using T = std::decay_t<decltype(e)>;

            if constexpr (std::is_same_v<T, KernelEvent>) {
                // Print KernelEvent fields
                std::cout
                    << "[KernelEvent] "
                    << "kernel=" << e.kernel_name
                    << ", device=" << e.device_id
                    << ", timestamp=" << e.timestamp_ns
                    << ", duration=" << e.duration_ns
                    << " ns\n";
            }
            else if constexpr (std::is_same_v<T, MemoryEvent>) {
                // Print MemoryEvent fields
                std::cout
                    << "[MemoryEvent] "
                    << "operation=" << e.operation
                    << ", device=" << e.device_id
                    << ", bytes=" << e.bytes
                    << ", timestamp=" << e.timestamp_ns
                    << " ns\n";
            }
        },
        event
    );
}

Writing distributed-profiler/src/console_sink.cpp


### test pipeline

In [26]:
%%writefile distributed-profiler/tests/test_basic_pipeline.cpp

#include "synthetic_event_source.hpp"
#include "simple_text_serializer.hpp"
#include "console_sink.hpp"

#include <cstddef>
#include <iostream>
#include <string>
#include <vector>

std::string bytes_to_string(const std::vector<std::byte>& bytes) {
    std::string result;
    result.reserve(bytes.size());

    for (std::byte b : bytes) {
        result.push_back(
            static_cast<char>(b)
        );
    }

    return result;
}

int main() {
    SyntheticEventSource source;
    SimpleTextSerializer serializer;
    ConsoleSink sink;

    std::size_t event_count = 0;

    while (auto event = source.next_event()) {

        ++event_count;

        std::cout
            << "\n=== Event "
            << event_count
            << " ===\n";

        // 1. Send the event to the sink
        sink.write(*event);

        // 2. Serialize the same event
        auto bytes =
            serializer.serialize(*event);

        // 3. Convert the byte buffer back to text
        //    only for validation/debugging.
        std::string serialized_text =
            bytes_to_string(bytes);

        std::cout
            << "Serialized: "
            << serialized_text
            << '\n';
    }

    std::cout
        << "\nTotal events processed: "
        << event_count
        << '\n';

    if (event_count != 4) {
        std::cerr
            << "Basic pipeline: FAIL\n";

        return 1;
    }

    std::cout
        << "Basic pipeline: PASS\n";

    return 0;
}

Writing distributed-profiler/tests/test_basic_pipeline.cpp


In [27]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/synthetic_event_source.cpp \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/console_sink.cpp \
    distributed-profiler/tests/test_basic_pipeline.cpp \
    -o test_basic_pipeline

In [28]:
!./test_basic_pipeline


=== Event 1 ===
[KernelEvent] kernel=gemm_kernel, device=0, timestamp=1000, duration=200 ns
Serialized: KERNEL|1000|200|0|gemm_kernel

=== Event 2 ===
[MemoryEvent] operation=H2D, device=0, bytes=4096, timestamp=1500 ns
Serialized: MEMORY|1500|0|4096|H2D

=== Event 3 ===
[KernelEvent] kernel=softmax_kernel, device=0, timestamp=2000, duration=150 ns
Serialized: KERNEL|2000|150|0|softmax_kernel

=== Event 4 ===
[MemoryEvent] operation=D2H, device=0, bytes=8192, timestamp=2300 ns
Serialized: MEMORY|2300|0|8192|D2H

Total events processed: 4
Basic pipeline: PASS


## CMake Build Validation

The final step in this notebook is to build the project using CMake rather than manually invoking `g++` for each test.

This validates that the codebase can be compiled as a structured C++ project with reusable source files, shared headers, and multiple test executables.

The current project contains:

```text
distributed-profiler/
├── include/
│   ├── profile_event.hpp
│   ├── event_source.hpp
│   ├── serializer.hpp
│   ├── data_sink.hpp
│   ├── synthetic_event_source.hpp
│   ├── simple_text_serializer.hpp
│   └── console_sink.hpp
│
├── src/
│   ├── synthetic_event_source.cpp
│   ├── simple_text_serializer.cpp
│   └── console_sink.cpp
│
├── tests/
│   ├── test_profile_event.cpp
│   ├── test_synthetic_event_source.cpp
│   ├── test_simple_text_serializer.cpp
│   └── test_basic_pipeline.cpp
│
└── CMakeLists.txt
```

The CMake build will:

- configure the project for C++20
- expose the `include/` directory to all targets
- compile the reusable implementation files into a shared project library
- build multiple test executables
- verify that the basic profiling pipeline works without manually specifying all compiler commands

This provides a cleaner foundation for later notebooks, where additional components such as queues, networking, storage, and benchmarks will be added to the same project structure.

In [29]:
%%writefile distributed-profiler/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

project(
    distributed_profiler
    VERSION 0.1.0
    LANGUAGES CXX
)

set(CMAKE_CXX_STANDARD 20)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

add_library(
    profiler_core
    src/synthetic_event_source.cpp
    src/simple_text_serializer.cpp
    src/console_sink.cpp
)

target_include_directories(
    profiler_core
    PUBLIC
        ${PROJECT_SOURCE_DIR}/include
)

target_compile_options(
    profiler_core
    PRIVATE
        -Wall
        -Wextra
        -Wpedantic
)

add_executable(
    test_profile_event
    tests/test_profile_event.cpp
)

target_link_libraries(
    test_profile_event
    PRIVATE
        profiler_core
)

add_executable(
    test_synthetic_event_source
    tests/test_synthetic_event_source.cpp
)

target_link_libraries(
    test_synthetic_event_source
    PRIVATE
        profiler_core
)

add_executable(
    test_simple_text_serializer
    tests/test_simple_text_serializer.cpp
)

target_link_libraries(
    test_simple_text_serializer
    PRIVATE
        profiler_core
)

add_executable(
    test_basic_pipeline
    tests/test_basic_pipeline.cpp
)

target_link_libraries(
    test_basic_pipeline
    PRIVATE
        profiler_core
)

Writing distributed-profiler/CMakeLists.txt


In [34]:
!cmake \
    -S distributed-profiler \
    -B distributed-profiler/build \
    -G Ninja

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /content/distributed-profiler/build


In [35]:
!ls distributed-profiler/build

build.ninja  CMakeCache.txt  CMakeFiles  cmake_install.cmake


In [38]:
!cmake --build distributed-profiler/build --verbose

Change Dir: '/content/distributed-profiler/build'

Run Build Command(s): /usr/bin/ninja -v
[1/12] /usr/bin/c++  -I/content/distributed-profiler/include -std=c++20 -Wall -Wextra -Wpedantic -MD -MT CMakeFiles/profiler_core.dir/src/console_sink.cpp.o -MF CMakeFiles/profiler_core.dir/src/console_sink.cpp.o.d -o CMakeFiles/profiler_core.dir/src/console_sink.cpp.o -c /content/distributed-profiler/src/console_sink.cpp
[2/12] /usr/bin/c++  -I/content/distributed-profiler/include -std=c++20 -Wall -Wextra -Wpedantic -MD -MT CMakeFiles/profiler_core.dir/src/simple_text_serializer.cpp.o -MF CMakeFiles/profiler_core.dir/src/simple_text_serializer.cpp.o.d -o CMakeFiles/profiler_core.dir/src/simple_text_serializer.cpp.o -c /content/distributed-profiler/src/simple_text_serializer.cpp
[3/12] /usr/bin/c++  -I/content/distributed-profiler/include -std=c++20 -Wall -Wextra -Wpedantic -MD -MT CMakeFiles/profiler_core.dir/src/synthetic_event_source.cpp.o -MF CMakeFiles/profiler_core.dir/src/synthetic_event_s

In [40]:
!find distributed-profiler/build -maxdepth 1 -type f -executable -print

distributed-profiler/build/test_synthetic_event_source
distributed-profiler/build/test_simple_text_serializer
distributed-profiler/build/test_basic_pipeline
distributed-profiler/build/test_profile_event


In [42]:
!./distributed-profiler/build/test_profile_event
!./distributed-profiler/build/test_synthetic_event_source
!./distributed-profiler/build/test_simple_text_serializer
!./distributed-profiler/build/test_basic_pipeline

[KernelEvent] device=0, kernel=gemm_kernel, timestamp=1000, duration=200 ns
[MemoryEvent] device=0, operation=H2D, bytes=4096, timestamp=1500 ns
[KernelEvent] kernel=gemm_kernel, device=0, timestamp=1000, duration=200 ns
[MemoryEvent] operation=H2D, device=0, bytes=4096, timestamp=1500 ns
[KernelEvent] kernel=softmax_kernel, device=0, timestamp=2000, duration=150 ns
[MemoryEvent] operation=D2H, device=0, bytes=8192, timestamp=2300 ns
Total events: 4
No more events: PASS
Kernel serialized: KERNEL|1000|200|0|gemm_kernel
Memory serialized: MEMORY|1500|0|4096|H2D
SimpleTextSerializer: PASS

=== Event 1 ===
[KernelEvent] kernel=gemm_kernel, device=0, timestamp=1000, duration=200 ns
Serialized: KERNEL|1000|200|0|gemm_kernel

=== Event 2 ===
[MemoryEvent] operation=H2D, device=0, bytes=4096, timestamp=1500 ns
Serialized: MEMORY|1500|0|4096|H2D

=== Event 3 ===
[KernelEvent] kernel=softmax_kernel, device=0, timestamp=2000, duration=150 ns
Serialized: KERNEL|2000|150|0|softmax_kernel

=== Event